# Excel与列式文件

学习目标：读写多工作表 Excel 与 Parquet 文件，保留业务编号和必要索引，并检查日期、缺失值及 dtype 的往返结果。

前置知识：文件读写、dtype、缺失值、表格索引。

运行环境：Python 3.12、pandas 3.0、openpyxl 3.1、PyArrow 25.0。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

配套脚本：本章没有外部脚本；配套数据位于 date/。

（1）[19-station-readings.xlsx](date/19-station-readings.xlsx)：六条自制站点测量记录，用于练习读取已有工作簿、保留编号和解释缺失值。

除该配套输入外，其余示例仍在单元内构造小表。生成的往返实验文件放在当前目录下的 TemporaryDirectory 中，退出对应 with 后自动清理；配套输入只读、不删除，读写句柄先于目录关闭。后续单元沿用首次导入的 pd、Path、TemporaryDirectory 和 assert_frame_equal。

## 1 写入并读取多工作表

把订单和门店清单交给使用表格软件的同事时，可以放入同一工作簿的不同工作表。ExcelWriter 管理整个输出工作簿，to_excel 的 sheet_name 指定表名；index=False 表示不把 pandas 行索引写成额外一列。

下面写出三个订单与两个门店，再用 ExcelFile 打开工作簿，查看工作表名称并读取内容。订单编号作为文本保存和读取，units 的单位为件。assert_frame_equal 用于比较值、标签、顺序和 dtype；一致时不输出。

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from pandas.testing import assert_frame_equal

orders = pd.DataFrame(
    {"order_id": ["001", "010", "103"], "store": ["S1", "S2", "S1"], "units": [2, 3, 1]}
)
stores = pd.DataFrame({"store": ["S1", "S2"], "city": ["北京", "上海"]})

with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    folder = Path(temp_dir)
    path = folder / "orders.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        orders.to_excel(writer, sheet_name="订单", index=False)
        stores.to_excel(writer, sheet_name="门店", index=False)

    with pd.ExcelFile(path, engine="openpyxl") as workbook:
        print(workbook.sheet_names)  # ['订单', '门店']。
        loaded_orders = pd.read_excel(workbook, sheet_name="订单", dtype={"order_id": "str"})
        loaded_stores = pd.read_excel(workbook, sheet_name="门店")

    print(loaded_orders)  # 001、010、103 原样保留，销量为 2、3、1。
    print(loaded_stores)  # S1 对应北京，S2 对应上海。
    print(loaded_orders.shape, loaded_stores.shape)  # (3, 3) (2, 2)。
    assert_frame_equal(orders, loaded_orders)
    assert_frame_equal(stores, loaded_stores)

print(folder.exists())  # False，工作簿已关闭，临时目录及文件已删除。

['订单', '门店']
  order_id store  units
0      001    S1      2
1      010    S2      3
2      103    S1      1
  store city
0    S1   北京
1    S2   上海
(3, 3) (2, 2)
False


## 2 引擎与工作表选择

pandas 提供表格接口，引擎负责具体格式的读写。本章显式选择引擎，避免依赖自动选择规则。

| 名称 | 中文名称／含义 | 本章用途 |
| --- | --- | --- |
| openpyxl | Excel 工作簿读写库 | 读取和写入 .xlsx 文件 |
| PyArrow | Arrow 数据与文件工具库 | 读写 Parquet 和 Feather |

read_excel 的 sheet_name 默认是 0，表示第一个工作表；名称或单个位置返回 DataFrame，列表或 None 返回字典，None 表示全部工作表。ExcelFile 适合在同一次打开中读取多个表，退出 with 后关闭。

下面复用上一节的 orders 和 stores，在新的临时目录中比较读取方式。ExcelWriter 默认 mode="w"，目标已存在时会覆盖；这里每次都使用新临时文件。

In [2]:
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    path = Path(temp_dir) / "sheets.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        orders.to_excel(writer, sheet_name="订单", index=False)
        stores.to_excel(writer, sheet_name="门店", index=False)

    with pd.ExcelFile(path, engine="openpyxl") as workbook:
        first = pd.read_excel(workbook, dtype={"order_id": "str"})
        selected = pd.read_excel(workbook, sheet_name=["门店"])
        all_sheets = pd.read_excel(workbook, sheet_name=None, dtype={"order_id": "str"})

    print(first.shape)  # (3, 3)，默认读取第一个工作表。
    print(type(selected).__name__, list(selected))  # dict ['门店']。
    print(list(all_sheets))  # ['订单', '门店']。
    print(all_sheets["订单"].equals(first))  # True。

(3, 3)
dict ['门店']
['订单', '门店']
True


## 3 保留编号中的前导零

编号即使只含数字，也可能不是数量。read_excel 默认推断 dtype，可能把文本编号 001 读成整数 1；读取时显式指定 string 可以保留文件里存储的文本。

如果单元格原本存的是数值 1，转成字符串只能得到“1”，无法凭空恢复原编号的位数。显示格式与存储值应分开考虑。

In [3]:
source = pd.DataFrame({"text_id": ["001", "010"], "numeric_id": [1, 10]})
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    path = Path(temp_dir) / "identifiers.xlsx"
    source.to_excel(path, engine="openpyxl", index=False)
    inferred = pd.read_excel(path, engine="openpyxl")
    explicit = pd.read_excel(
        path, engine="openpyxl", dtype={"text_id": "string", "numeric_id": "string"}
    )

    print(inferred["text_id"].tolist(), inferred["text_id"].dtype)  # [1, 10] int64。
    print(explicit["text_id"].tolist())  # ['001', '010']。
    print(explicit["numeric_id"].tolist())  # ['1', '10']，不自动补成三位编号。
    print(explicit["text_id"].dtype, explicit["text_id"].dtype.storage)  # string pyarrow。

[1, 10] int64
['001', '010']
['1', '10']
string pyarrow


## 4 读取日期文本

文件中的日期可能是 Excel 日期单元格，也可能只是文本。对于格式已约定的日期文本，parse_dates 指定列，date_format 指定解释方式。

下面的 %Y、%m、%d 分别表示四位年、月、日；输入约定为“年/月/日”。读取后检查 dtype，而不是只看打印出来像不像日期。

In [4]:
source = pd.DataFrame({"order_id": ["001", "010"], "day": ["2026/09/01", "2026/09/02"]})
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    path = Path(temp_dir) / "dates.xlsx"
    source.to_excel(path, engine="openpyxl", index=False)
    parsed = pd.read_excel(
        path,
        engine="openpyxl",
        dtype={"order_id": "string"},
        parse_dates=["day"],
        date_format="%Y/%m/%d",
    )

    print(parsed)  # 两条编号不变，日期为 2026-09-01、2026-09-02。
    print(parsed.dtypes)  # order_id 为 string；本次 day 为 datetime64[us]，无时区。
    print(parsed["day"].isna().tolist())  # [False, False]。

  order_id        day
0      001 2026-09-01
1      010 2026-09-02
order_id            string
day         datetime64[us]
dtype: object
[False, False]


输入有无效日期时，不能把启用 parse_dates 当成“所有行已经解析成功”。需要逐行识别失败项时，先保留日期文本，再用 to_datetime 的 errors="coerce" 把解析失败变成 NaT，并保留原字段供核查。

下面分别区分“原本缺失”和“有文本但无效”，避免把二者合并成同一个原因。

In [5]:
source = pd.DataFrame(
    {"order_id": ["001", "010", "103"], "raw_day": ["2026/09/01", "2026/02/30", None]}
)
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    path = Path(temp_dir) / "invalid-dates.xlsx"
    source.to_excel(path, engine="openpyxl", index=False)
    loaded = pd.read_excel(
        path, engine="openpyxl", dtype={"order_id": "string", "raw_day": "string"}
    )
    loaded["day"] = pd.to_datetime(loaded["raw_day"], format="%Y/%m/%d", errors="coerce")
    loaded["invalid_day"] = loaded["raw_day"].notna() & loaded["day"].isna()

    print(loaded)  # 三行均保留；第二、三行 day 为 NaT，原日期字段仍在。
    print(loaded["invalid_day"].tolist())  # [False, True, False]。

  order_id     raw_day        day  invalid_day
0      001  2026/09/01 2026-09-01        False
1      010  2026/02/30        NaT         True
2      103        <NA>        NaT        False
[False, True, False]


## 5 缺失值与 Excel 类型往返

Excel 保存单元格值，不能自动保存完整的 pandas dtype 约定。可空整数、分类集合和分类顺序等信息，应在读取时按已知规则恢复，并检查实际结果。

下面写入文本编号、可空整数、有序分类和无时区日期。对比默认推断与显式指定类型；这组日期按整天记录，不涉及高精度时间保存。

In [6]:
grade_dtype = pd.CategoricalDtype(categories=["低", "中", "高"], ordered=True)
source = pd.DataFrame(
    {
        "code": pd.Series(["001", "010", "103"], dtype="string"),
        "units": pd.Series([2, None, 1], dtype="Int64"),
        "grade": pd.Categorical(["低", "高", "低"], dtype=grade_dtype),
        "day": pd.to_datetime(["2026-09-01", "2026-09-02", "2026-09-03"]),
    }
)
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    path = Path(temp_dir) / "types.xlsx"
    source.to_excel(path, engine="openpyxl", index=False)
    inferred = pd.read_excel(path, engine="openpyxl")
    restored = pd.read_excel(
        path,
        engine="openpyxl",
        dtype={"code": "string", "units": "Int64", "grade": grade_dtype},
        parse_dates=["day"],
    )

    print(inferred.dtypes)  # code 为 int64，units 为 float64，grade 为 str。
    print(restored.dtypes)  # 恢复为 string、Int64、category；day 为 datetime64[us]。
    print(restored["grade"].cat.categories.tolist(), restored["grade"].cat.ordered)
    # ['低', '中', '高'] True，未出现的“中”也来自显式类别约定。
    assert_frame_equal(source, restored)
    print(restored.equals(source))  # True，本例值、标签及类型均恢复。

code              int64
units           float64
grade               str
day      datetime64[us]
dtype: object
code             string
units             Int64
grade          category
day      datetime64[us]
dtype: object
['低', '中', '高'] True
True


读取 Excel 时，默认缺失标记包括空字符串和文本 NA。若 NA 是业务中的合法文本，可以关闭默认标记，再明确指定哪些输入算缺失。

下面用 keep_default_na=False 保留文本 NA，并用 na_values=[""] 把空单元格视为缺失。to_excel 默认把缺失写为空白，本例的空串与缺失写入后无法再靠默认读取规则区分；有这种要求时应另加状态列。

In [7]:
source = pd.DataFrame({"row": [1, 2, 3, 4], "note": ["NA", "", None, "北京"]})
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    path = Path(temp_dir) / "missing.xlsx"
    source.to_excel(path, engine="openpyxl", index=False)
    default = pd.read_excel(path, engine="openpyxl", dtype={"note": "string"})
    explicit = pd.read_excel(
        path,
        engine="openpyxl",
        dtype={"note": "string"},
        keep_default_na=False,
        na_values=[""],
    )

    print(default["note"].tolist())  # [<NA>, <NA>, <NA>, '北京']。
    print(explicit["note"].tolist())  # ['NA', <NA>, <NA>, '北京']。
    print(explicit.shape)  # (4, 2)，row 列使各条记录都有非空内容。

[<NA>, <NA>, <NA>, '北京']
['NA', <NA>, <NA>, '北京']
(4, 2)


### 5.1 读取已有的测量工作簿

打开配套的 date/19-station-readings.xlsx，可以先查看 readings 工作表，再用下方代码读取。它是六条自制教学数据，不是实际气象观测；每行表示一个站点的一次采样，station_id 是保留三位的文本编号，sample_no 是站点内的采样序号，temperature_c 的单位是摄氏度（°C）。

温度空白表示未取得读数，0 表示有效的零摄氏度，二者不能混为一谈。note 中 NA 是合法的原始备注文本，sensor_offline 表示模拟的传感器离线，freezing_point 表示零度示例，ok 表示普通记录；空白备注表示未填写。读取时关闭默认缺失词表，只把空白单元格判为缺失，并显式指定编号、采样序号、温度和备注的类型。

In [8]:
station_readings = pd.read_excel(
    "date/19-station-readings.xlsx",
    sheet_name="readings",
    engine="openpyxl",
    dtype={
        "station_id": "string", "sample_no": "Int64",
        "temperature_c": "Float64", "note": "string",
    },
    keep_default_na=False,
    na_values=[""],
)
print(station_readings)  # 六行四列；001、010 保留前导零，NA 备注保留原文。
print(station_readings.dtypes)  # string、Int64、Float64、string。
assert station_readings["station_id"].tolist() == ["001", "001", "010", "010", "103", "103"]
assert station_readings["temperature_c"].isna().tolist() == [False, True, False, False, False, False]
assert station_readings.loc[2, "temperature_c"] == 0
assert station_readings.loc[0, "note"] == "NA"
assert pd.isna(station_readings.loc[3, "note"])
print(station_readings["temperature_c"].count())  # 5 个有效温度，包括零摄氏度。

  station_id  sample_no  temperature_c            note
0        001          1           21.5              NA
1        001          2           <NA>  sensor_offline
2        010          1            0.0  freezing_point
3        010          2           20.0            <NA>
4        103          1           19.5              ok
5        103          2           20.5              ok
station_id        string
sample_no          Int64
temperature_c    Float64
note              string
dtype: object
5


## 6 索引是否写入 Excel

to_excel 默认 index=True，把行标签写入工作表。业务标识如果放在索引里，需明确决定是否保存；index=False 会省略它，而不会把它自动移成普通列。

下面为索引指定名称 row_id。读回时先作为普通列检查，再用 set_index 恢复，避免在读取合并单元格时隐含采用 index_col 的缺失前向填充规则。

In [9]:
source = pd.DataFrame({"units": [2, 3]}, index=pd.Index(["R1", "R2"], name="row_id"))
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    path = Path(temp_dir) / "index.xlsx"
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        source.to_excel(writer, sheet_name="保留索引", index=True)
        source.to_excel(writer, sheet_name="省略索引", index=False)
    with pd.ExcelFile(path, engine="openpyxl") as workbook:
        kept = pd.read_excel(workbook, sheet_name="保留索引")
        omitted = pd.read_excel(workbook, sheet_name="省略索引")

    restored = kept.set_index("row_id")
    print(kept.columns.tolist())  # ['row_id', 'units']。
    print(omitted.columns.tolist(), omitted.index.tolist())  # ['units'] [0, 1]。
    print(restored.index.tolist(), restored.index.name)  # ['R1', 'R2'] row_id。
    assert_frame_equal(source, restored)

['row_id', 'units']


['units'] [0, 1]
['R1', 'R2'] row_id


## 7 Excel 日期的时区边界

Excel 日期单元格不支持时区。把带时区的 pandas 日期直接写入会失败；直接去掉时区也会丢失解释时间所需的信息。

本例采用明确的交换约定：把 UTC 时间写成带 +00:00 偏移的文本，读回后按 ISO 8601 规则解析为 UTC。文本单元格不再是 Excel 原生日期；需要在表格软件中计算日期时，应另行约定无时区值的含义。

In [10]:
source = pd.DataFrame({"at": pd.to_datetime(["2026-09-01 08:00"], utc=True)})
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    folder = Path(temp_dir)
    try:
        with pd.ExcelWriter(folder / "timezone.xlsx", engine="openpyxl") as writer:
            source.to_excel(writer, index=False)
    except ValueError as error:
        assert "timezones" in str(error)
        print(type(error).__name__)  # ValueError，Excel 日期不支持时区。
    else:
        raise AssertionError("预期带时区日期不能直接写入 Excel")

    exported = pd.DataFrame({"at_text": source["at"].astype("string")})
    text_path = folder / "timezone-text.xlsx"
    exported.to_excel(text_path, engine="openpyxl", index=False)
    loaded = pd.read_excel(text_path, engine="openpyxl", dtype={"at_text": "string"})
    recovered = pd.to_datetime(loaded["at_text"], format="ISO8601", utc=True)
    print(loaded["at_text"].tolist())  # ['2026-09-01 08:00:00+00:00']。
    print(recovered.dtype)  # 本次为 datetime64[us, UTC]。
    print(recovered.tolist() == source["at"].tolist())  # True，同一个 UTC 时间点。

ValueError
['2026-09-01 08:00:00+00:00']
datetime64[us, UTC]
True


## 8 Parquet 文件往返

Parquet 是面向表格数据的列式二进制格式，支持压缩和按列读取。它适合程序间的数据交换；使用 to_parquet 和 read_parquet，本章均指定 engine="pyarrow"。

下面保存带业务索引的小表，包含可空整数、可空布尔、有序分类、显式 string 和默认 str。PyArrow 会保存 pandas 类型相关元数据，但其他程序生成的文件或不同读取后端不一定恢复相同 dtype，仍需检查。

In [11]:
typed = pd.DataFrame(
    {
        "code": pd.Series(["001", None, "010"], dtype="string"),
        "units": pd.Series([1, None, 3], dtype="Int64"),
        "active": pd.Series([True, None, False], dtype="boolean"),
        "grade": pd.Categorical(["低", "高", "低"], categories=["低", "中", "高"], ordered=True),
        "label": ["甲", "乙", None],
    }
)
typed.index = pd.Index(["R1", "R2", "R3"], name="row_id")
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    folder = Path(temp_dir)
    path = folder / "typed.parquet"
    typed.to_parquet(path, engine="pyarrow", index=True)
    loaded = pd.read_parquet(path, engine="pyarrow")

    print(loaded)  # 三行五列；R2 的 code、units、active 缺失，R3 的 label 缺失。
    print(loaded.dtypes)  # 依次为 string、Int64、boolean、category、str。
    print(loaded.index.tolist(), loaded.index.name)  # ['R1', 'R2', 'R3'] row_id。
    print(loaded["grade"].cat.categories.tolist(), loaded["grade"].cat.ordered)
    # ['低', '中', '高'] True；完整类别集合和顺序保留。
    print(loaded["code"].dtype.storage, loaded["label"].dtype.storage)  # pyarrow pyarrow。
    assert_frame_equal(typed, loaded)

print(folder.exists())  # False，Parquet 文件与临时目录已清理。

        code  units  active grade label
row_id                                 
R1       001      1    True     低     甲
R2      <NA>   <NA>    <NA>     高     乙
R3       010      3   False     低   NaN


code        string
units        Int64
active     boolean
grade     category
label          str
dtype: object
['R1', 'R2', 'R3'] row_id
['低', '中', '高'] True
pyarrow pyarrow
False


## 9 Parquet 的索引保存选项

index=True 保存索引，index=False 不保存。默认 index=None 也保留索引信息，其中 RangeIndex 可只在元数据中保存起点、终点与步长，其他索引通常作为物理列保存。

下面对比业务索引被保留或省略时的结果。选择 index=False 前，应确认需要的业务标识已经是普通列。

In [12]:
source = pd.DataFrame({"units": [2, 3]}, index=pd.Index(["R1", "R2"], name="row_id"))
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    folder = Path(temp_dir)
    kept_path = folder / "with-index.parquet"
    plain_path = folder / "without-index.parquet"
    source.to_parquet(kept_path, engine="pyarrow", index=True)
    source.to_parquet(plain_path, engine="pyarrow", index=False)
    kept = pd.read_parquet(kept_path, engine="pyarrow")
    plain = pd.read_parquet(plain_path, engine="pyarrow")

    print(kept.index.tolist(), kept.index.name)  # ['R1', 'R2'] row_id。
    print(plain.index.tolist(), plain.index.name)  # [0, 1] None，业务标签未保存。
    print(plain.columns.tolist())  # ['units']，不自动补回 row_id 列。
    assert_frame_equal(source, kept)

    ranged = pd.DataFrame({"units": [2, 3]}, index=pd.RangeIndex(10, 14, 2, name="row_id"))
    range_path = folder / "range-index.parquet"
    ranged.to_parquet(range_path, engine="pyarrow", index=None)
    range_back = pd.read_parquet(range_path, engine="pyarrow")
    print(range_back.index)  # RangeIndex(start=10, stop=14, step=2, name='row_id')。
    assert_frame_equal(ranged, range_back)

['R1', 'R2'] row_id
[0, 1] None
['units']
RangeIndex(start=10, stop=14, step=2, name='row_id')


## 10 选学：列选择与压缩

只分析部分字段时，read_parquet 的 columns 指定要读取的数据列。compression 控制 Parquet 的压缩方式；None 表示不压缩，下面对照 gzip。压缩不会替代 dtype 和索引约定，实际体积收益取决于数据，不根据几行样本承诺压缩率。

下面复用 typed；自定义索引仍由文件中的 pandas 元数据恢复。Excel 的 usecols 也能限制读取列，但工作簿与列式格式的存储方式不同。

In [13]:
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    folder = Path(temp_dir)
    raw_path = folder / "plain.parquet"
    gzip_path = folder / "gzip.parquet"
    typed.to_parquet(raw_path, engine="pyarrow", index=True, compression=None)
    typed.to_parquet(gzip_path, engine="pyarrow", index=True, compression="gzip")
    raw_back = pd.read_parquet(raw_path, engine="pyarrow")
    gzip_back = pd.read_parquet(gzip_path, engine="pyarrow")
    subset = pd.read_parquet(gzip_path, engine="pyarrow", columns=["code", "units"])

    print(subset)  # 只含 code、units 两列，R1、R2、R3 仍为行标签。
    print(subset.shape, subset.columns.tolist())  # (3, 2) ['code', 'units']。
    assert_frame_equal(raw_back, gzip_back)
    assert_frame_equal(subset, typed[["code", "units"]])
    print(raw_back.equals(gzip_back))  # True，压缩设置没有改变本例数据。

        code  units
row_id             
R1       001      1
R2      <NA>   <NA>
R3       010      3
(3, 2) ['code', 'units']
True


Excel 中只需要编号和销量时，可用 usecols 选择列，用 nrows 限制数据行数；nrows 不包含表头。下面使用开篇的 orders，仅取前两条订单的两个字段。

In [14]:
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    path = Path(temp_dir) / "selected-columns.xlsx"
    orders.to_excel(path, engine="openpyxl", index=False)
    selected = pd.read_excel(
        path,
        engine="openpyxl",
        usecols=["order_id", "units"],
        nrows=2,
        dtype={"order_id": "string"},
    )
    print(selected)  # 两行两列：001 对应 2，010 对应 3，不含 store 列。
    print(selected.shape)  # (2, 2)。

  order_id  units
0      001      2
1      010      3
(2, 2)


## 11 选学：Feather 与其他格式入口

Feather 是基于 Arrow 的二进制表格文件格式，常用于保存中间数据。本例使用 Feather V2 和 PyArrow，由 to_feather、read_feather 读写；这里没有 engine 参数。V2 支持 zstd 等压缩方式。

为使业务标识明确可见，先把 typed 的 row_id 索引转成普通列，写入带默认行索引的表，读回后再恢复业务索引。

In [15]:
exported = typed.reset_index()
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    folder = Path(temp_dir)
    path = folder / "intermediate.feather"
    exported.to_feather(path, version=2, compression="zstd")
    loaded = pd.read_feather(path)
    small = pd.read_feather(path, columns=["row_id", "units"])
    restored = loaded.set_index("row_id")

    print(loaded.shape, loaded.columns.tolist())
    # (3, 6)，row_id 成为第一列，其后为 code、units、active、grade、label。
    print(small)  # 三行两列，units 仍为 Int64，第二行缺失。
    print(restored.dtypes)  # 恢复后类型与 typed 一致。
    assert_frame_equal(typed, restored)

print(folder.exists())  # False，Feather 文件及目录已清理。

(3, 6) ['row_id', 'code', 'units', 'active', 'grade', 'label']
  row_id  units
0     R1      1
1     R2   <NA>
2     R3      3
code        string
units        Int64
active     boolean
grade     category
label          str
dtype: object
False


其他格式按交换对象和已有系统选择，不仅看扩展名。本章只给出入口，不安装额外引擎。

| 格式 | 中文名称／含义 | pandas 读取入口与条件 |
| --- | --- | --- |
| JSON | 对象与数组组织的文本数据 | read_json；先约定表格组织方式和类型解析 |
| ORC | 列式二进制数据 | read_orc；使用 PyArrow，需核对支持的类型与读取选项 |
| HDF5 | 分层数据文件 | read_hdf；pandas 对应的 HDFStore 数据需要 PyTables 支持 |

这些格式的文件往返不在本章实验范围；当前示例验证的是 .xlsx、Parquet 和 Feather V2。

## 本章小结

（1）ExcelWriter 管理写入，ExcelFile 管理多工作表读取，with 负责关闭资源。工作表选择、索引保存和引擎都应明确。

（2）编号先确定是否是文本；日期按明确格式解析。空串、缺失标记、时区和 pandas dtype 不能仅靠工作表外观判断。

（3）Parquet 保存列式数据，index 参数决定索引保存方式。即使有类型元数据，也要检查实际值、标签、分类集合、缺失和 dtype。

（4）列选择减少读取的字段，压缩改变文件表示。Feather 可用于中间表交换；不同文件来源和读取后端仍需单独核对往返约定。

## 练习

（1）把订单与状态说明写入同一 .xlsx 的两张工作表，再用 ExcelFile 读取。新增约束为“编号必须保留三位，status 中的 NA 是合法文本”，选择读取参数并说明理由，不能把它误认为缺失。

In [16]:
exercise_orders = pd.DataFrame({"order_id": ["007", "020"], "status": ["NA", "完成"]})
exercise_notes = pd.DataFrame({"status": ["NA", "完成"], "meaning": ["待确认", "已完成"]})
# 在此使用 TemporaryDirectory(dir=".")、ExcelWriter 和 ExcelFile 完成往返。
# 检查：两张工作表都存在，编号和 NA 原文不变，退出 with 后文件已清理。

（2）先预测下面读回的索引与列名，再运行。随后要求改变为“R1、R2 是必须跨文件保留的业务键”，给出两种保存方案并说明各自读回后的表结构。

In [17]:
exercise = pd.DataFrame({"units": [5, 8]}, index=pd.Index(["R1", "R2"], name="row_id"))
with TemporaryDirectory(dir=".", prefix="pandas19-") as temp_dir:
    path = Path(temp_dir) / "exercise.parquet"
    exercise.to_parquet(path, engine="pyarrow", index=False)
    back = pd.read_parquet(path, engine="pyarrow")
    print(back.index)
    print(back.columns.tolist())
# 运行后核对预测，再修改写入方式；比较保留索引与先转成普通列的选择。

RangeIndex(start=0, stop=2, step=1)
['units']


（3）把下面带缺失和有序分类的小表保存为 Parquet，检查往返后的类型、类别集合和顺序。随后只读 units 列；再改用 Excel 交换，说明需要额外保存或指定哪些类型约定。

In [18]:
exercise = pd.DataFrame(
    {
        "units": pd.Series([2, None, 6], dtype="Int64"),
        "grade": pd.Categorical(["中", "低", "中"], categories=["低", "中", "高"], ordered=True),
    }
)
# 在此在临时目录中使用 pyarrow 引擎完成写读，检查未出现的“高”类别是否保留。
# 检查：全部读取为 (3, 2)，列选择为 (3, 1)，缺失仍在第二行。

（4）将原日期文本写入 Excel 后读回，再按“年/月/日”解析并标记无效输入，保留原字段。若接收方改为要求带 UTC 时区的时间点，应选择 Excel 日期、带偏移文本还是列式文件？说明选择理由与读回检查方法。

In [19]:
exercise = pd.DataFrame(
    {"id": ["001", "002", "003"], "raw_day": ["2026/09/01", "2026/13/01", None]}
)
# 在此先读成文本，再显式解析；标记“原文非缺失但解析失败”的记录。
# 检查：第二行属于解析失败，第三行属于原本缺失，三条记录都保留。

（5）重新读取配套测量工作簿，只取 station_id、sample_no 和 temperature_c 三列。保留前导零、温度缺失和有效的零值，找出没有温度读数的站点与采样序号。说明为什么不能用温度是否为零来识别缺失；不要修改原始工作簿。

In [20]:
# 在此使用 read_excel 的 usecols、dtype 和缺失值参数重新读取配套文件。
# 检查：形状为 (6, 3)；缺失记录是站点 001 的第 2 次采样。
# 站点 010 的第 1 次采样温度为 0，应保留为有效读数。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [IO tools](https://pandas.pydata.org/docs/user_guide/io.html) 的 Excel files：Reading Excel files、Dtype specifications、Writing Excel files，以及 Parquet、Feather；[read_excel](https://pandas.pydata.org/docs/reference/api/pandas.read_excel.html) 的 sheet_name、dtype、parse_dates、date_format、keep_default_na、na_values、index_col、usecols、nrows；[ExcelFile](https://pandas.pydata.org/docs/reference/api/pandas.ExcelFile.html) 的工作簿解析与引擎；[ExcelWriter](https://pandas.pydata.org/docs/reference/api/pandas.ExcelWriter.html) 的上下文管理、mode；[to_excel](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_excel.html) 的多工作表、index 和 na_rep；[to_datetime](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) 的 format、errors、utc；[to_parquet](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_parquet.html) 的 engine、compression、index 与分类元数据；[read_parquet](https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html) 的 columns、读取后端；[to_feather](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_feather.html)、[read_feather](https://pandas.pydata.org/docs/reference/api/pandas.read_feather.html) 的引擎转交、默认索引与列选择；[assert_frame_equal](https://pandas.pydata.org/docs/reference/api/pandas.testing.assert_frame_equal.html) 的类型、标签、类别检查；[安装与可选依赖](https://pandas.pydata.org/docs/getting_started/install.html) 的 Excel、Parquet 和 HDF5；其他格式入口：[read_json](https://pandas.pydata.org/docs/reference/api/pandas.read_json.html)、[read_orc](https://pandas.pydata.org/docs/reference/api/pandas.read_orc.html)、[read_hdf](https://pandas.pydata.org/docs/reference/api/pandas.read_hdf.html)。 |
| Apache Arrow 官方文档（25.0.1） | [Pandas Integration](https://arrow.apache.org/docs/python/pandas.html) 的 Handling pandas Indexes、Nullable types：索引与 pandas 类型元数据、非 pandas 来源的条件；[Feather File Format](https://arrow.apache.org/docs/python/feather.html) 的 V2、Using Compression；[parquet.write_table](https://arrow.apache.org/docs/python/generated/pyarrow.parquet.write_table.html) 的 compression 与 schema 存储。 |
| Apache Parquet 官方文档 | [Overview](https://parquet.apache.org/docs/overview/) 的列式格式、压缩与数据交换用途。 |
| openpyxl 官方文档（稳定页面标识 3.1.3，本机 3.1.5） | [Dates and Times](https://openpyxl.readthedocs.io/en/stable/datetime.html) 的 Excel 日期表示及 Timezones：原生日期单元格不支持时区。 |
| Python 官方文档（Python 3.12） | [TemporaryDirectory](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory) 的 dir、上下文退出清理及 Windows 未关闭文件的清理条件。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[io](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/io.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |